# Picking the right detector

Univariate detectors don't all catch the same kinds of shift. This notebook walks through three
canonical scenarios and shows which detector flags each one and which stays quiet.

Rule of thumb:

| Shift type | Strongest signals |
|---|---|
| Mean shift | KS, Wasserstein, PSI |
| Variance shift | Wasserstein, CVM (KS misses pure variance shift in symmetric data) |
| Tail/heavy-tail shift | CVM, Wasserstein (PSI may underweight depending on binning) |

In [ ]:
import numpy as np

from drift_control import UnifiedDriftDetector

# Permutation-calibrated detectors get a smaller n_permutations to keep
# the notebook fast; the calibration-free detectors don't take that kwarg.
DETECTOR_KWARGS = {
    "psi": {},
    "ks": {},
    "cvm": {},
    "js": {},
    "wasserstein": {"n_permutations": 80, "random_state": 0},
}


def score(ref, cur):
    return {
        m: UnifiedDriftDetector(method=m, **kw).detect_drift(ref, cur)
        for m, kw in DETECTOR_KWARGS.items()
    }


def show(results):
    for method, out in results.items():
        p = "n/a" if out.p_value is None else f"{out.p_value:.3f}"
        flag = "DRIFT" if out.drift else "     "
        print(f"  {method:>12}  {flag}  score={out.score:.4f}  p={p}")


rng = np.random.default_rng(42)

## Scenario 1: No drift (control)

Same distribution on both sides. Detectors should mostly stay quiet.

In [ ]:
ref = rng.normal(0, 1, size=600)
cur = rng.normal(0, 1, size=600)
show(score(ref, cur))

## Scenario 2: Mean shift

Current distribution is shifted +0.8 standard deviations. Almost every detector should fire.

In [ ]:
ref = rng.normal(0, 1, size=600)
cur = rng.normal(0.8, 1, size=600)
show(score(ref, cur))

## Scenario 3: Variance shift only

Same mean, wider spread (sigma 1 → 1.8). Variance-sensitive detectors (Wasserstein, CVM)
catch this more reliably than KS on symmetric data.

In [ ]:
ref = rng.normal(0, 1, size=600)
cur = rng.normal(0, 1.8, size=600)
show(score(ref, cur))

## Scenario 4: Heavy-tail (t-distribution)

Reference is Normal; current is Student-t with df=3. Same first two moments approximately,
but heavier tails. CVM and Wasserstein generally win here.

In [ ]:
ref = rng.normal(0, 1, size=600)
cur = rng.standard_t(df=3, size=600)
show(score(ref, cur))

## Scenario 5: Categorical shift

Three categories with different mixing weights. Use the categorical detectors directly.

In [ ]:
ref_cat = rng.choice(["a", "b", "c"], size=600, p=[0.5, 0.4, 0.1])
cur_cat = rng.choice(["a", "b", "c"], size=600, p=[0.1, 0.3, 0.6])

for method in ("chi2cat", "tvdcat"):
    out = UnifiedDriftDetector(method=method).detect_drift(ref_cat, cur_cat)
    p = "n/a" if out.p_value is None else f"{out.p_value:.3f}"
    flag = "DRIFT" if out.drift else "     "
    print(f"  {method:>10}  {flag}  score={out.score:.4f}  p={p}")

## Takeaway

There's no single "best" detector — pick based on the shift type you're worried about, or
use `EnsembleDriftDetector` to vote across several at once (see `04_ensemble_and_multiple_testing.ipynb`).